This notebook adds team box scores for 2024/25 to team_metrics_dataset.csv

In [4]:
import pandas as pd
from nba_api.stats.endpoints import boxscoretraditionalv3
from nba_api.stats.endpoints import boxscoreadvancedv3
from nba_api.stats.endpoints import boxscorefourfactorsv3
from nba_api.stats.endpoints import boxscoremiscv3
from nba_api.stats.endpoints import boxscorescoringv3
from functools import reduce
import time
import numpy as np

In [2]:
# Import data
full_nba_data = pd.read_csv("full_nba_data.csv", dtype={'GAME_ID': object}) # basic box scores for 1946/47 - 2024/25
team_metrics_dataset = pd.read_csv("team_metrics_dataset.csv", dtype={'GAME_ID': object}) # complete team box scores for 1996/97 - 2023/24

In [3]:
# filter for games from the 2024/25 season (all columns in team_metrics_dataset that are not boxscore data)
basic_boxscores_post2024 = full_nba_data[full_nba_data["YEAR"] == 2024][["GAME_ID","YEAR","GAME_DATE","SEASON_TYPE","HOME_TEAM_ID","AWAY_TEAM_ID","HOME_WL","AWAY_WL"]]

In [12]:
# functions to call the nba_api and get cleaned boxscore data
def get_boxscore_traditional(game_id):
    """
    Call nba_api and get cleaned boxscore traditional stats for a single game
    https://github.com/swar/nba_api/blob/master/docs/nba_api/stats/endpoints/boxscoretraditionalv3.md
    https://www.nba.com/stats/teams/boxscores-traditional
    
    Args:
        game_id: 10 digit string representing an individual game

    Returns:
        boxscore_traditional_df: dataframe containing boxscore traditiional stats for a single game
    """
    # call api and get dataframe
    boxscore_traditional = boxscoretraditionalv3.BoxScoreTraditionalV3(game_id = game_id) 
    boxscore_traditional_df = boxscore_traditional.get_data_frames()[2]

    # remove and rename columns to match team_metrics_dataset.csv
    boxscore_traditional_df.drop(['teamCity', 'teamName', 'teamTricode', 'teamSlug'], axis=1, inplace=True)

    boxscore_traditional_df.columns = ['GAME_ID', 'TEAM_ID', 'TRAD_MIN', 'TRAD_FGM', 'TRAD_FGA', 'TRAD_FG_PCT', 'TRAD_3PM', 'TRAD_3PA', 'TRAD_3P_PCT', 'TRAD_FTM', 'TRAD_FTA', 'TRAD_FT_PCT', 'TRAD_OREB', 'TRAD_DREB', 'TRAD_REB', 'TRAD_AST', 'TRAD_STL', 'TRAD_BLK', 'TRAD_TOV', 'TRAD_PF', 'TRAD_PTS', 'TRAD_PLUS_MINUS']

    return boxscore_traditional_df

def get_boxscore_advanced(game_id):
    """
    Call nba_api and get cleaned boxscore advanced stats for a single game
    https://github.com/swar/nba_api/blob/master/docs/nba_api/stats/endpoints/boxscoreadvancedv3.md
    https://www.nba.com/stats/teams/boxscores-advanced
    
    Args:
        game_id: 10 digit string representing an individual game

    Returns:
        boxscore_advanced_df: dataframe containing boxscore advanced stats for a single game
    """
    # call api and get dataframe
    boxscore_advanced = boxscoreadvancedv3.BoxScoreAdvancedV3(game_id = game_id) 
    boxscore_advanced_df = boxscore_advanced.get_data_frames()[1]

    # remove and rename columns to match team_metrics_dataset.csv
    boxscore_advanced_df.drop(['teamCity', 'teamName', 'teamTricode', 'teamSlug', 'minutes', 'estimatedOffensiveRating', 'estimatedDefensiveRating', 'estimatedNetRating', 'turnoverRatio', 'usagePercentage', 'estimatedUsagePercentage', 'estimatedPace', 'pacePer40', 'possessions', 'estimatedTeamTurnoverPercentage', 'offensiveReboundPercentage', 'effectiveFieldGoalPercentage'], axis=1, inplace=True)
    boxscore_advanced_df.columns = ['GAME_ID', 'TEAM_ID', 'ADV_OFFRTG', 'ADV_DEFRTG', 'ADV_NETRTG', 'ADV_AST_PCT', 'ADV_AST_TO', 'ADV_AST_RATIO',  'ADV_DREB_PCT', 'ADV_REB_PCT', 'ADV_TS_PCT', 'ADV_PACE', 'ADV_PIE']

    return boxscore_advanced_df

def get_boxscore_fourfactors(game_id):
    """
    Call nba_api and get cleaned boxscore four factors stats for a single game
    https://github.com/swar/nba_api/blob/master/docs/nba_api/stats/endpoints/boxscorefourfactorsv3.md
    https://www.nba.com/stats/teams/boxscores-four-factors
    
    Args:
        game_id: 10 digit string representing an individual game

    Returns:
        boxscore_fourfactors_df: dataframe containing boxscore four factors stats for a single game
    """
    # call api and get dataframe
    boxscore_fourfactors = boxscorefourfactorsv3.BoxScoreFourFactorsV3(game_id = game_id) 
    boxscore_fourfactors_df = boxscore_fourfactors.get_data_frames()[1]

    # remove and rename columns to match team_metrics_dataset.csv
    boxscore_fourfactors_df.drop(['teamCity', 'teamName', 'teamTricode', 'teamSlug', 'minutes'], axis=1, inplace=True)
    boxscore_fourfactors_df.columns = ['GAME_ID', 'TEAM_ID', 'FF_EFG_PCT', 'FF_FTA_RATE', 'FF_TOV_PCT', 'FF_OREB_PCT', 'FF_OPP_EFG_PCT', 'FF_OPP_FTA_RATE', 'FF_OPP_TOV_PCT', 'FF_OPP_OREB_PCT']

    return boxscore_fourfactors_df

def get_boxscore_misc(game_id):
    """
    Call nba_api and get cleaned boxscore misc stats for a single game
    https://github.com/swar/nba_api/blob/master/docs/nba_api/stats/endpoints/boxscorefourfactorsv3.md
    https://www.nba.com/stats/teams/boxscores-four-factors
    
    Args:
        game_id: 10 digit string representing an individual game

    Returns:
        boxscore_misc_df: dataframe containing boxscore misc stats for a single game
    """
    # call api and get dataframe
    boxscore_misc = boxscoremiscv3.BoxScoreMiscV3(game_id = game_id) 
    boxscore_misc_df = boxscore_misc.get_data_frames()[1]

    # remove and rename columns to match team_metrics_dataset.csv
    boxscore_misc_df.drop(['teamCity', 'teamName', 'teamTricode', 'teamSlug', 'minutes', 'blocks', 'blocksAgainst', 'foulsPersonal', 'foulsDrawn'], axis=1, inplace=True)
    boxscore_misc_df.columns = ['GAME_ID', 'TEAM_ID', 'MISC_PTS_OFF_TO', 'MISC_2ND_PTS', 'MISC_FBPS', 'MISC_PITP', 'MISC_OPP_PTS_OFF_TO', 'MISC_OPP_2ND_PTS', 'MISC_OPP_FBPS', 'MISC_OPP_PITP']

    return boxscore_misc_df

def get_boxscore_scoring(game_id):
    """
    Call nba_api and get cleaned boxscore scoring stats for a single game
    https://github.com/swar/nba_api/blob/master/docs/nba_api/stats/endpoints/boxscorefourfactorsv3.md
    https://www.nba.com/stats/teams/boxscores-scoring
    
    Args:
        game_id: 10 digit string representing an individual game

    Returns:
        boxscore_fourfactors_df: dataframe containing boxscore scoringstats for a single game
    """
    # call api and get dataframe
    boxscore_scoring = boxscorescoringv3.BoxScoreScoringV3(game_id = game_id) 
    boxscore_scoring_df = boxscore_scoring.get_data_frames()[1]

    # remove and rename columns to match team_metrics_dataset.csv
    boxscore_scoring_df.drop(['teamCity', 'teamName', 'teamTricode', 'teamSlug', 'minutes'], axis=1, inplace=True)
    boxscore_scoring_df.columns = ['GAME_ID', 'TEAM_ID', 'SCOR_PCTFGA_2PT', 'SCOR_PCTFGA_3PT', 'SCOR_PCTPTS_2PT', 'SCOR_PCTPTS_2PT_MR', 'SCOR_PCTPTS_3PT', 'SCOR_PCTPTS_FBPS', 'SCOR_PCTPTS_FT', 'SCOR_PCTPTS_OFF_TO', 'SCOR_PCTPTS_PITP', 'SCOR_2FGM__PCTAST', 'SCOR_2FGM__PCTUAST', 'SCOR_3FGM__PCTAST', 'SCOR_3FGM__PCTUAST', 'SCOR_FGM__PCTAST', 'SCOR_FGM__PCTUAST']

    return boxscore_scoring_df

In [5]:
def get_team_metrics_boxscores(one_game_basic_boxscores, sleep_time = 20):
    """
    Gets the team metrics for a single game
    
    Args:
        one_game_basic_boxscores (Series): one row of basic boxscores

    Returns:
        one_game_team_metrics (Dataframe): two rows of team metrics (unique game, two teams)
    """
    game_id = one_game_basic_boxscores['GAME_ID']

    boxscores_traditional = get_boxscore_traditional(game_id=game_id)
    time.sleep(sleep_time)
    boxscore_advanced = get_boxscore_advanced(game_id=game_id)
    time.sleep(sleep_time)
    boxscore_fourfactors = get_boxscore_fourfactors(game_id=game_id)
    time.sleep(sleep_time)
    boxscore_misc = get_boxscore_misc(game_id=game_id)
    time.sleep(sleep_time)
    boxscore_scoring = get_boxscore_scoring(game_id=game_id)
    time.sleep(sleep_time)

    boxscore_dfs = [boxscores_traditional, boxscore_advanced, boxscore_fourfactors, boxscore_misc, boxscore_scoring]

    merge_func = lambda left_df, right_df: pd.merge(left_df, right_df, on=['GAME_ID','TEAM_ID'])

    one_game_team_metrics = reduce(merge_func, boxscore_dfs)

    one_game_team_metrics = one_game_team_metrics.merge(one_game_basic_boxscores.to_frame().T, how="left", on='GAME_ID')

    return one_game_team_metrics

def get_side_and_winloss(one_game_team):
    """
    Gets the SIDE and WINLOSS for a single game and team

    Args:
        one_game_team (Series): one row of team metrics (unique game+team)

    Returns:
        side (str): HOME/AWAY depending on if the team is the home/away team
        winloss (str): W/L depending on if the team won/loss the game
    """
    if one_game_team['TEAM_ID'] == one_game_team['HOME_TEAM_ID']:
        side = "HOME"
        winloss = one_game_team['HOME_WL']
    elif one_game_team['TEAM_ID'] == one_game_team['AWAY_TEAM_ID']:
        side = "AWAY"
        winloss = one_game_team['AWAY_WL']
    else:
        raise ValueError(f"Team {one_game_team['TEAM_ID']} is not {one_game_team['HOME_TEAM_ID']} nor {one_game_team['AWAY_TEAM_ID']}")
    
    return side, winloss

def create_side_and_winloss_lists(one_game_team_metrics):
    """
    Creates lists for SIDE and WINLOSS for a single game

    Args:
        one_game (Dataframe): two rows of team metrics (unique game, two teams)

    Returns:
        side_list (lst): list of HOME/AWAY depending on if the team is the home/away team
        winloss_list (lst): list of W/L depending on if the team won/loss the game
    """
    side_list, winloss_list = [], []

    for _, one_game_team in one_game_team_metrics.iterrows():
        side, winloss = get_side_and_winloss(one_game_team)

        side_list.append(side)
        winloss_list.append(winloss)

    return side_list, winloss_list

In [ ]:
def create_team_metrics_df(basic_boxscores_post2024):
   """
   Create team metrics dataframe
   
   Args: 
      basic_boxscores_post2024 (Dataframe): basic boxscores
   
   Returns:
      team_metrics (Dataframe): team metrics
   """
   team_metrics_list = []

   for index, one_game_basic_boxscores in basic_boxscores_post2024.iterrows():
      try:
         # get beginning team metrics data for the game (two rows)
         one_game_team_metrics = get_team_metrics_boxscores(one_game_basic_boxscores, sleep_time = 10)

         # create columns SIDE (HOME/AWAY) and WINLOSS (W/L)
         SIDE, WINLOSS = create_side_and_winloss_lists(one_game_team_metrics)
         one_game_team_metrics['SIDE'], one_game_team_metrics['WINLOSS'] = SIDE, WINLOSS

         # drop unnecessary columns
         one_game_team_metrics.drop(['HOME_WL', 'AWAY_WL'], axis=1, inplace=True)

         # append final team metrics for the game to team metrics list
         team_metrics_list.append(one_game_team_metrics)

         print(f"Team metrics gotten for index {index}")
   
      except Exception as e:
         print(f"Failed at index {index}: {e}")

      # pause every 10 games for 30 seconds
      if index % 10 == 0:
         time.sleep(30)
      
   if team_metrics_list:
      team_metrics = pd.concat(team_metrics_list, ignore_index = True)
   else:
      team_metrics = pd.DataFrame()

   return team_metrics

In [12]:
# Split into 10 DataFrames
boxscores_df1, boxscores_df2, boxscores_df3, boxscores_df4, boxscores_df5, boxscores_df6, boxscores_df7, boxscores_df8, boxscores_df9, boxscores_df10 = np.array_split(basic_boxscores_post2024, 10)

In [14]:
team_metrics_df1 = create_team_metrics_df(boxscores_df1)

Team metrics gotten for index 70532
Team metrics gotten for index 70533
Team metrics gotten for index 70534
Team metrics gotten for index 70535
Team metrics gotten for index 70536
Team metrics gotten for index 70537
Team metrics gotten for index 70538
Team metrics gotten for index 70539
Team metrics gotten for index 70540
Team metrics gotten for index 70541
Team metrics gotten for index 70542
Team metrics gotten for index 70543
Team metrics gotten for index 70544
Team metrics gotten for index 70545
Team metrics gotten for index 70546
Team metrics gotten for index 70547
Team metrics gotten for index 70548
Team metrics gotten for index 70549
Team metrics gotten for index 70550
Team metrics gotten for index 70551
Team metrics gotten for index 70552
Team metrics gotten for index 70553
Team metrics gotten for index 70554
Team metrics gotten for index 70555
Team metrics gotten for index 70556
Team metrics gotten for index 70557
Team metrics gotten for index 70558
Team metrics gotten for inde

In [ ]:
team_metrics_df2 = create_team_metrics_df(boxscores_df2)

Team metrics gotten for index 70664
Team metrics gotten for index 70665
Team metrics gotten for index 70666
Team metrics gotten for index 70667
Team metrics gotten for index 70668
Team metrics gotten for index 70669
Team metrics gotten for index 70670
Team metrics gotten for index 70671
Team metrics gotten for index 70672
Team metrics gotten for index 70673
Team metrics gotten for index 70674
Team metrics gotten for index 70675
Team metrics gotten for index 70676
Team metrics gotten for index 70677
Team metrics gotten for index 70678
Team metrics gotten for index 70679
Team metrics gotten for index 70680
Team metrics gotten for index 70681
Team metrics gotten for index 70682
Team metrics gotten for index 70683
Team metrics gotten for index 70684
Team metrics gotten for index 70685
Team metrics gotten for index 70686
Team metrics gotten for index 70687
Team metrics gotten for index 70688
Team metrics gotten for index 70689
Team metrics gotten for index 70690
Team metrics gotten for inde

In [ ]:
team_metrics_df3 = create_team_metrics_df(boxscores_df3)

Team metrics gotten for index 70796
Team metrics gotten for index 70797
Team metrics gotten for index 70798
Team metrics gotten for index 70799
Team metrics gotten for index 70800
Team metrics gotten for index 70801
Team metrics gotten for index 70802
Team metrics gotten for index 70803
Team metrics gotten for index 70804
Team metrics gotten for index 70805
Team metrics gotten for index 70806
Team metrics gotten for index 70807
Team metrics gotten for index 70808
Team metrics gotten for index 70809
Team metrics gotten for index 70810
Team metrics gotten for index 70811
Team metrics gotten for index 70812
Team metrics gotten for index 70813
Team metrics gotten for index 70814
Team metrics gotten for index 70815
Team metrics gotten for index 70816
Team metrics gotten for index 70817
Team metrics gotten for index 70818
Team metrics gotten for index 70819
Team metrics gotten for index 70820
Team metrics gotten for index 70821
Team metrics gotten for index 70822
Team metrics gotten for inde

In [ ]:
team_metrics_df4 = create_team_metrics_df(boxscores_df4)

Team metrics gotten for index 70928
Team metrics gotten for index 70929
Team metrics gotten for index 70930
Team metrics gotten for index 70931
Team metrics gotten for index 70932
Team metrics gotten for index 70933
Team metrics gotten for index 70934
Team metrics gotten for index 70935
Team metrics gotten for index 70936
Team metrics gotten for index 70937
Team metrics gotten for index 70938
Team metrics gotten for index 70939
Team metrics gotten for index 70940
Team metrics gotten for index 70941
Team metrics gotten for index 70942
Team metrics gotten for index 70943
Team metrics gotten for index 70944
Team metrics gotten for index 70945
Team metrics gotten for index 70946
Team metrics gotten for index 70947
Team metrics gotten for index 70948
Team metrics gotten for index 70949
Team metrics gotten for index 70950
Team metrics gotten for index 70951
Team metrics gotten for index 70952
Team metrics gotten for index 70953
Team metrics gotten for index 70954
Team metrics gotten for inde

In [ ]:
team_metrics_df5 = create_team_metrics_df(boxscores_df5)

Team metrics gotten for index 71060
Team metrics gotten for index 71061
Team metrics gotten for index 71062
Team metrics gotten for index 71063
Team metrics gotten for index 71064
Team metrics gotten for index 71065
Team metrics gotten for index 71066
Team metrics gotten for index 71067
Team metrics gotten for index 71068
Team metrics gotten for index 71069
Team metrics gotten for index 71070
Team metrics gotten for index 71071
Team metrics gotten for index 71072
Team metrics gotten for index 71073
Team metrics gotten for index 71074
Team metrics gotten for index 71075
Team metrics gotten for index 71076
Team metrics gotten for index 71077
Team metrics gotten for index 71078
Team metrics gotten for index 71079
Team metrics gotten for index 71080
Team metrics gotten for index 71081
Failed at index 71082: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=30)
Failed at index 71083: Expecting value: line 1 column 1 (char 0)
Failed at index 71084: Expecting 

In [ ]:
team_metrics_df6 = create_team_metrics_df(boxscores_df6)

Team metrics gotten for index 71192
Team metrics gotten for index 71193
Team metrics gotten for index 71194
Team metrics gotten for index 71195
Team metrics gotten for index 71196
Team metrics gotten for index 71197
Team metrics gotten for index 71198
Team metrics gotten for index 71199
Team metrics gotten for index 71200
Team metrics gotten for index 71201
Team metrics gotten for index 71202
Team metrics gotten for index 71203
Team metrics gotten for index 71204
Team metrics gotten for index 71205
Team metrics gotten for index 71206
Team metrics gotten for index 71207
Team metrics gotten for index 71208
Team metrics gotten for index 71209
Team metrics gotten for index 71210
Team metrics gotten for index 71211
Team metrics gotten for index 71212
Team metrics gotten for index 71213
Team metrics gotten for index 71214
Team metrics gotten for index 71215
Team metrics gotten for index 71216
Team metrics gotten for index 71217
Team metrics gotten for index 71218
Team metrics gotten for inde

In [ ]:
team_metrics_df7 = create_team_metrics_df(boxscores_df7)

Team metrics gotten for index 71324
Team metrics gotten for index 71325
Team metrics gotten for index 71326
Team metrics gotten for index 71327
Team metrics gotten for index 71328
Team metrics gotten for index 71329
Team metrics gotten for index 71330
Team metrics gotten for index 71331
Team metrics gotten for index 71332
Team metrics gotten for index 71333
Team metrics gotten for index 71334
Team metrics gotten for index 71335
Team metrics gotten for index 71336
Team metrics gotten for index 71337
Team metrics gotten for index 71338
Team metrics gotten for index 71339
Team metrics gotten for index 71340
Team metrics gotten for index 71341
Team metrics gotten for index 71342
Team metrics gotten for index 71343
Team metrics gotten for index 71344
Team metrics gotten for index 71345
Team metrics gotten for index 71346
Team metrics gotten for index 71347
Team metrics gotten for index 71348
Team metrics gotten for index 71349
Team metrics gotten for index 71350
Team metrics gotten for inde

In [ ]:
team_metrics_df8 = create_team_metrics_df(boxscores_df8)

Team metrics gotten for index 71456
Team metrics gotten for index 71457
Team metrics gotten for index 71458
Team metrics gotten for index 71459
Team metrics gotten for index 71460
Team metrics gotten for index 71461
Team metrics gotten for index 71462
Team metrics gotten for index 71463
Team metrics gotten for index 71464
Team metrics gotten for index 71465
Team metrics gotten for index 71466
Team metrics gotten for index 71467
Team metrics gotten for index 71468
Team metrics gotten for index 71469
Team metrics gotten for index 71470
Team metrics gotten for index 71471
Team metrics gotten for index 71472
Team metrics gotten for index 71473
Team metrics gotten for index 71474
Team metrics gotten for index 71475
Team metrics gotten for index 71476
Team metrics gotten for index 71477
Team metrics gotten for index 71478
Team metrics gotten for index 71479
Team metrics gotten for index 71480
Team metrics gotten for index 71481
Team metrics gotten for index 71482
Team metrics gotten for inde

In [ ]:
team_metrics_df9 = create_team_metrics_df(boxscores_df9)

Team metrics gotten for index 71588
Team metrics gotten for index 71589
Team metrics gotten for index 71590
Team metrics gotten for index 71591
Team metrics gotten for index 71592
Team metrics gotten for index 71593
Team metrics gotten for index 71594
Team metrics gotten for index 71595
Team metrics gotten for index 71596
Team metrics gotten for index 71597
Team metrics gotten for index 71598
Team metrics gotten for index 71599
Team metrics gotten for index 71600
Team metrics gotten for index 71601
Team metrics gotten for index 71602
Team metrics gotten for index 71603
Team metrics gotten for index 71604
Team metrics gotten for index 71605
Team metrics gotten for index 71606
Team metrics gotten for index 71607
Team metrics gotten for index 71608
Team metrics gotten for index 71609
Team metrics gotten for index 71610
Team metrics gotten for index 71611
Team metrics gotten for index 71612
Team metrics gotten for index 71613
Team metrics gotten for index 71614
Team metrics gotten for inde

In [ ]:
team_metrics_df10 = create_team_metrics_df(boxscores_df10)

Team metrics gotten for index 71720
Team metrics gotten for index 71721
Team metrics gotten for index 71722
Team metrics gotten for index 71723
Team metrics gotten for index 71724
Team metrics gotten for index 71725
Team metrics gotten for index 71726
Team metrics gotten for index 71727
Team metrics gotten for index 71728
Team metrics gotten for index 71729
Team metrics gotten for index 71730
Team metrics gotten for index 71731
Team metrics gotten for index 71732
Team metrics gotten for index 71733
Team metrics gotten for index 71734
Team metrics gotten for index 71735
Team metrics gotten for index 71736
Team metrics gotten for index 71737
Team metrics gotten for index 71738
Team metrics gotten for index 71739
Team metrics gotten for index 71740
Team metrics gotten for index 71741
Team metrics gotten for index 71742
Team metrics gotten for index 71743
Team metrics gotten for index 71744
Team metrics gotten for index 71745
Team metrics gotten for index 71746
Team metrics gotten for inde

In [ ]:
failed_indexes = [71082, 71083, 71084]
team_metrics_df11 = create_team_metrics_df(boxscores_df5.loc[failed_indexes])

Team metrics gotten for index 71082
Team metrics gotten for index 71083
Team metrics gotten for index 71084


In [93]:
team_metrics = pd.concat([
    team_metrics_df1, team_metrics_df2, team_metrics_df3, team_metrics_df4, team_metrics_df5,
    team_metrics_df6, team_metrics_df7, team_metrics_df8, team_metrics_df9, team_metrics_df10, team_metrics_df11
], ignore_index=True)


In [ ]:
team_metrics_unique_sorted = team_metrics[['GAME_ID']].drop_duplicates().sort_values(by='GAME_ID').reset_index(drop=True)
basic_boxscores_post2024_sorted = basic_boxscores_post2024[['GAME_ID']].sort_values(by="GAME_ID").reset_index(drop=True)
missing_game_ids = basic_boxscores_post2024_sorted[~basic_boxscores_post2024_sorted['GAME_ID'].isin(team_metrics_unique_sorted['GAME_ID'])]
missing_game_ids_list = missing_game_ids['GAME_ID'].tolist()

dataframes = [
    team_metrics_df1, team_metrics_df2, team_metrics_df3, team_metrics_df4, team_metrics_df5,
    team_metrics_df6, team_metrics_df7, team_metrics_df8, team_metrics_df9, team_metrics_df10, team_metrics_df11
]

num_games = 0

for i, df in enumerate(dataframes, start=1):
    num_games += len(df)
    num_missing = sum(df['GAME_ID'].isin(missing_game_ids_list))
    print(f"team_metrics_df{i}: number IDs = {len(df)/2},missing IDs = {num_missing}")

print(num_games)

team_metrics_df1: number IDs = 132.0,missing IDs = 0
team_metrics_df2: number IDs = 132.0,missing IDs = 0
team_metrics_df3: number IDs = 132.0,missing IDs = 0
team_metrics_df4: number IDs = 132.0,missing IDs = 0
team_metrics_df5: number IDs = 129.0,missing IDs = 0
team_metrics_df6: number IDs = 132.0,missing IDs = 0
team_metrics_df7: number IDs = 132.0,missing IDs = 0
team_metrics_df8: number IDs = 132.0,missing IDs = 0
team_metrics_df9: number IDs = 132.0,missing IDs = 0
team_metrics_df10: number IDs = 132.0,missing IDs = 0
team_metrics_df11: number IDs = 3.0,missing IDs = 0
2640


In [105]:
team_metrics_full = pd.concat([team_metrics_dataset, team_metrics], ignore_index=True)

In [106]:
team_metrics_full.to_csv('team_metrics.csv', index=False)

In [5]:
team_metrics_full = pd.read_csv("team_metrics.csv", dtype={'GAME_ID': object}) # complete team box scores for 1996/97 - 2024/25

In [54]:
team_metrics_full.isnull().sum()

GAME_ID               0
TEAM_ID               0
GAME_DATE             0
SEASON_TYPE           0
YEAR                  0
                     ..
SCOR_2FGM__PCTUAST    0
SCOR_3FGM__PCTAST     0
SCOR_3FGM__PCTUAST    0
SCOR_FGM__PCTAST      0
SCOR_FGM__PCTUAST     0
Length: 71, dtype: int64

In [6]:
missing_values_count = team_metrics_full.isnull().sum()
team_metrics_full_columns = team_metrics_full.columns
team_metrics_full_columns[missing_values_count > 0]

Index(['TRAD_PTS', 'TRAD_FGM', 'TRAD_FGA', 'TRAD_FG_PCT', 'TRAD_3PM',
       'TRAD_3PA', 'TRAD_3P_PCT', 'TRAD_FTM', 'TRAD_FTA', 'TRAD_FT_PCT',
       'TRAD_OREB', 'TRAD_DREB', 'TRAD_REB', 'TRAD_AST', 'TRAD_TOV',
       'TRAD_STL', 'TRAD_BLK', 'TRAD_PF', 'TRAD_PLUS_MINUS', 'TRAD_MIN',
       'ADV_OFFRTG', 'ADV_DEFRTG', 'ADV_NETRTG', 'ADV_AST_PCT', 'ADV_AST_TO',
       'ADV_AST_RATIO', 'ADV_DREB_PCT', 'ADV_REB_PCT', 'ADV_TS_PCT',
       'ADV_PACE', 'ADV_PIE'],
      dtype='object')

In [ ]:
# Filter rows where ADV_OFFRTG is missing
missing_adv_stats = team_metrics_full[team_metrics_full['ADV_OFFRTG'].isna()]

# Extract game IDs for where ADV_OFFRTG is missing
missing_adv_stats_IDs = missing_adv_stats['GAME_ID'].dropna().unique()

missing_adv_stats_list = []

for index, game_id in enumerate(missing_adv_stats_IDs):
    try:
        # get advanced boxscores
        missing_adv_stats_list.append(get_boxscore_advanced(game_id))

        print(f"Team metrics gotten for game ID = {game_id}, index = {index}")

    except Exception as e:
        print(f"Failed at game ID = {game_id}, index = {index}: {e}")

    # pause every 20 games for 10 seconds
    if index % 20 == 0:
        time.sleep(10)
        
if missing_adv_stats_list:
    team_metrics = pd.concat(missing_adv_stats_list, ignore_index = True)
else:
    team_metrics = pd.DataFrame()

# Define the columns to fill
adv_cols = [
    'ADV_OFFRTG', 'ADV_DEFRTG', 'ADV_NETRTG', 'ADV_AST_PCT', 'ADV_AST_TO',
    'ADV_AST_RATIO', 'ADV_DREB_PCT', 'ADV_REB_PCT', 'ADV_TS_PCT',
    'ADV_PACE', 'ADV_PIE'
]

# Merge the dataframes
team_metrics2 = pd.merge(
    team_metrics_full,
    team_metrics[['GAME_ID', 'TEAM_ID'] + adv_cols],
    on=['GAME_ID', 'TEAM_ID'],
    how='left',
    suffixes=('', '_from_metrics')
)

# Fill missing values from team_metrics
for col in adv_cols:
    team_metrics2[col] = team_metrics2[col].combine_first(team_metrics2[f"{col}_from_metrics"])
    team_metrics2.drop(columns=[f"{col}_from_metrics"], inplace=True)

team_metrics2.to_csv('team_metrics2.csv', index=False)

Team metrics gotten for game ID = 0029600014, index = 0
Team metrics gotten for game ID = 0029600028, index = 1
Team metrics gotten for game ID = 0029600035, index = 2
Team metrics gotten for game ID = 0029600046, index = 3
Team metrics gotten for game ID = 0029600058, index = 4
Team metrics gotten for game ID = 0029600074, index = 5
Team metrics gotten for game ID = 0029600088, index = 6
Team metrics gotten for game ID = 0029600103, index = 7
Team metrics gotten for game ID = 0029600111, index = 8
Team metrics gotten for game ID = 0029600128, index = 9
Team metrics gotten for game ID = 0029600129, index = 10
Team metrics gotten for game ID = 0029600145, index = 11
Team metrics gotten for game ID = 0029600174, index = 12
Team metrics gotten for game ID = 0029600181, index = 13
Team metrics gotten for game ID = 0029600190, index = 14
Team metrics gotten for game ID = 0029600203, index = 15
Team metrics gotten for game ID = 0029600210, index = 16
Team metrics gotten for game ID = 0029600

In [ ]:
# Filter rows where TRAD_PTS is missing
missing_trad_stats = team_metrics_full[team_metrics_full['TRAD_PTS'].isna()]

# Extract game IDs for where TRAD_PTS is missing
missing_trad_stats_IDs = missing_trad_stats['GAME_ID'].dropna().unique()

missing_trad_stats_list = []

for index, game_id in enumerate(missing_trad_stats_IDs):
    try:
        # get advanced boxscores
        missing_trad_stats_list.append(get_boxscore_traditional(game_id))

        print(f"Team metrics gotten for game ID = {game_id}, index = {index}")

    except Exception as e:
        print(f"Failed at game ID = {game_id}, index = {index}: {e}")

    # pause every 20 games for 20 seconds
    if index % 20 == 0:
        time.sleep(20)
        
if missing_trad_stats_list:
    team_metrics = pd.concat(missing_trad_stats_list, ignore_index = True)
else:
    team_metrics = pd.DataFrame()


Team metrics gotten for game ID = 0020200001, index = 0
Team metrics gotten for game ID = 0020200002, index = 1
Team metrics gotten for game ID = 0020200003, index = 2
Team metrics gotten for game ID = 0020200004, index = 3
Team metrics gotten for game ID = 0020200005, index = 4
Team metrics gotten for game ID = 0020200006, index = 5
Team metrics gotten for game ID = 0020200007, index = 6
Team metrics gotten for game ID = 0020200008, index = 7
Team metrics gotten for game ID = 0020200009, index = 8
Team metrics gotten for game ID = 0020200010, index = 9
Team metrics gotten for game ID = 0020200011, index = 10
Team metrics gotten for game ID = 0020200012, index = 11
Team metrics gotten for game ID = 0020200013, index = 12
Team metrics gotten for game ID = 0020200014, index = 13
Team metrics gotten for game ID = 0020200015, index = 14
Team metrics gotten for game ID = 0020200016, index = 15
Team metrics gotten for game ID = 0020200017, index = 16
Team metrics gotten for game ID = 0020200

In [46]:

# Define the columns to fill
trad_cols = [
    'TRAD_PTS', 'TRAD_FGM', 'TRAD_FGA', 'TRAD_FG_PCT', 'TRAD_3PM',
       'TRAD_3PA', 'TRAD_3P_PCT', 'TRAD_FTM', 'TRAD_FTA', 'TRAD_FT_PCT',
       'TRAD_OREB', 'TRAD_DREB', 'TRAD_REB', 'TRAD_AST', 'TRAD_TOV',
       'TRAD_STL', 'TRAD_BLK', 'TRAD_PF', 'TRAD_PLUS_MINUS', 'TRAD_MIN'
]

# Merge the dataframes
team_metrics3 = pd.merge(
    team_metrics2,
    team_metrics[['GAME_ID', 'TEAM_ID'] + trad_cols],
    on=['GAME_ID', 'TEAM_ID'],
    how='left',
    suffixes=('', '_from_metrics')
)

# Fill missing values from team_metrics
for col in trad_cols:
    team_metrics3[col] = team_metrics3[col].combine_first(team_metrics3[f"{col}_from_metrics"])
    team_metrics3.drop(columns=[f"{col}_from_metrics"], inplace=True)

team_metrics3.to_csv('team_metrics3.csv', index=False)

In [58]:
team_metrics3[['GAME_ID']]

,GAME_ID
0,0020200460
1,0020200460
2,0020200460
3,0020200460
4,0020200460
...,...
73399,0020200460
73400,0020200460
73401,0020200460
73402,0020200460


In [ ]:
# Extract game IDs for where TRAD_PTS is still missing
still_missing_trad_stats_IDs = list(set(missing_trad_stats_IDs) - set(team_metrics['GAME_ID']))

still_missing_trad_stats_list = []

for index, game_id in enumerate(still_missing_trad_stats_IDs):
    try:
        # get advanced boxscores
        still_missing_trad_stats_list.append(get_boxscore_traditional(game_id))

        print(f"Team metrics gotten for game ID = {game_id}, index = {index}")

    except Exception as e:
        print(f"Failed at game ID = {game_id}, index = {index}: {e}")

    # pause every 10 games for 30 seconds
    if index % 10 == 0:
        time.sleep(30)
        
if still_missing_trad_stats_list:
    team_metrics = pd.concat(still_missing_trad_stats_list, ignore_index = True)
else:
    team_metrics = pd.DataFrame()

Team metrics gotten for game ID = 0020200489, index = 0
Team metrics gotten for game ID = 0020200472, index = 1
Team metrics gotten for game ID = 0020200569, index = 2
Team metrics gotten for game ID = 0040200313, index = 3
Team metrics gotten for game ID = 0040200222, index = 4
Team metrics gotten for game ID = 0020200563, index = 5
Team metrics gotten for game ID = 0020200517, index = 6
Team metrics gotten for game ID = 0040200153, index = 7
Team metrics gotten for game ID = 0040200146, index = 8
Team metrics gotten for game ID = 0040200233, index = 9
Team metrics gotten for game ID = 0020200577, index = 10
Team metrics gotten for game ID = 0020200518, index = 11
Team metrics gotten for game ID = 0020200901, index = 12
Team metrics gotten for game ID = 0020200558, index = 13
Team metrics gotten for game ID = 0020200596, index = 14
Team metrics gotten for game ID = 0040200141, index = 15
Team metrics gotten for game ID = 0020200334, index = 16
Team metrics gotten for game ID = 0040200

508

In [48]:
# Define the columns to fill
trad_cols = [
    'TRAD_PTS', 'TRAD_FGM', 'TRAD_FGA', 'TRAD_FG_PCT', 'TRAD_3PM',
       'TRAD_3PA', 'TRAD_3P_PCT', 'TRAD_FTM', 'TRAD_FTA', 'TRAD_FT_PCT',
       'TRAD_OREB', 'TRAD_DREB', 'TRAD_REB', 'TRAD_AST', 'TRAD_TOV',
       'TRAD_STL', 'TRAD_BLK', 'TRAD_PF', 'TRAD_PLUS_MINUS', 'TRAD_MIN'
]

# Merge the dataframes
team_metrics4 = pd.merge(
    team_metrics3,
    team_metrics[['GAME_ID', 'TEAM_ID'] + trad_cols],
    on=['GAME_ID', 'TEAM_ID'],
    how='left',
    suffixes=('', '_from_metrics')
)

# Fill missing values from team_metrics
for col in trad_cols:
    team_metrics4[col] = team_metrics4[col].combine_first(team_metrics4[f"{col}_from_metrics"])
    team_metrics4.drop(columns=[f"{col}_from_metrics"], inplace=True)

team_metrics4.to_csv('team_metrics4.csv', index=False)

In [60]:
team_metrics_full = pd.read_csv("team_metrics3.csv", dtype={'GAME_ID': object}) # complete team box scores for 1996/97 - 2024/25

C:\Users\jessa\AppData\Local\Temp\ipykernel_103152\2739214956.py:1: DtypeWarning: Columns (28) have mixed types. Specify dtype option on import or set low_memory=False.
  team_metrics_full = pd.read_csv("team_metrics3.csv", dtype={'GAME_ID': object}) # complete team box scores for 1996/97 - 2024/25


In [62]:
missing_values_count = team_metrics_full.isnull().sum()
team_metrics_full_columns = team_metrics_full.columns
team_metrics_full_columns[missing_values_count > 0]

Index(['TRAD_PTS', 'TRAD_FGM', 'TRAD_FGA', 'TRAD_FG_PCT', 'TRAD_3PM',
       'TRAD_3PA', 'TRAD_3P_PCT', 'TRAD_FTM', 'TRAD_FTA', 'TRAD_FT_PCT',
       'TRAD_OREB', 'TRAD_DREB', 'TRAD_REB', 'TRAD_AST', 'TRAD_TOV',
       'TRAD_STL', 'TRAD_BLK', 'TRAD_PF', 'TRAD_PLUS_MINUS', 'TRAD_MIN'],
      dtype='object')

In [ ]:
# Filter rows where TRAD_PTS is missing
missing_trad_stats = team_metrics_full[team_metrics_full['TRAD_PTS'].isna()]

# Extract game IDs for where TRAD_PTS is missing
missing_trad_stats_IDs = missing_trad_stats['GAME_ID'].dropna().unique()

missing_trad_stats_list = []
error_game_IDs = []
num_gotten, num_errored = 0, 0

for index, game_id in enumerate(missing_trad_stats_IDs):
    try:
        # get advanced boxscores
        missing_trad_stats_list.append(get_boxscore_traditional(game_id))
        num_gotten += 1

        print(f"Team metrics gotten for game ID = {game_id}, index = {index}")

    except Exception as e:
        # append game IDs
        error_game_IDs.append(game_id)
        num_errored += 1

        print(f"Failed at game ID = {game_id}, index = {index}: {e}")

    # pause every 10 games for 30 seconds
    if index % 10 == 0:
        time.sleep(30)
        
if missing_trad_stats_list:
    missing_trad_statsteam_metrics = pd.concat(missing_trad_stats_list, ignore_index = True)
else:
    missing_trad_statsteam_metrics = pd.DataFrame()



Team metrics gotten for game ID = 0020200001, index = 0
Team metrics gotten for game ID = 0020200002, index = 1
Team metrics gotten for game ID = 0020200003, index = 2
Team metrics gotten for game ID = 0020200004, index = 3
Team metrics gotten for game ID = 0020200005, index = 4
Team metrics gotten for game ID = 0020200006, index = 5
Team metrics gotten for game ID = 0020200007, index = 6
Team metrics gotten for game ID = 0020200008, index = 7
Team metrics gotten for game ID = 0020200009, index = 8
Team metrics gotten for game ID = 0020200010, index = 9
Team metrics gotten for game ID = 0020200011, index = 10
Team metrics gotten for game ID = 0020200012, index = 11
Team metrics gotten for game ID = 0020200013, index = 12
Team metrics gotten for game ID = 0020200014, index = 13
Team metrics gotten for game ID = 0020200015, index = 14
Team metrics gotten for game ID = 0020200016, index = 15
Team metrics gotten for game ID = 0020200017, index = 16
Team metrics gotten for game ID = 0020200

1942

In [72]:
missing_trad_stats_list2 = []
error_game_IDs2 = []
num_gotten2, num_errored2 = 0, 0

for index, game_id in enumerate(error_game_IDs):
    try:
        # get advanced boxscores
        missing_trad_stats_list2.append(get_boxscore_traditional(game_id))
        num_gotten2 += 1

        print(f"Team metrics gotten for game ID = {game_id}, index = {index}")

    except Exception as e:
        # append game IDs
        error_game_IDs2.append(game_id)
        num_errored2 += 1

        print(f"Failed at game ID = {game_id}, index = {index}: {e}")

    # pause every 10 games for 30 seconds
    if index % 10 == 0:
        time.sleep(30)
        
if missing_trad_stats_list2:
    missing_trad_statsteam_metrics2 = pd.concat(missing_trad_stats_list2, ignore_index = True)
else:
    missing_trad_statsteam_metrics2 = pd.DataFrame()

print(f"{num_gotten2 = }, {num_errored2 = }, {len(error_game_IDs) = }, {num_errored2+num_gotten2 = }")

Team metrics gotten for game ID = 0020200644, index = 0
Team metrics gotten for game ID = 0020200645, index = 1
Team metrics gotten for game ID = 0020200646, index = 2
Team metrics gotten for game ID = 0020200647, index = 3
Team metrics gotten for game ID = 0020200648, index = 4
Team metrics gotten for game ID = 0020200649, index = 5
Team metrics gotten for game ID = 0020200650, index = 6
Team metrics gotten for game ID = 0020200651, index = 7
Team metrics gotten for game ID = 0020200652, index = 8
Team metrics gotten for game ID = 0020200653, index = 9
Team metrics gotten for game ID = 0020200654, index = 10
Team metrics gotten for game ID = 0020200655, index = 11
Team metrics gotten for game ID = 0020200656, index = 12
Team metrics gotten for game ID = 0020200657, index = 13
Team metrics gotten for game ID = 0020200658, index = 14
Team metrics gotten for game ID = 0020200659, index = 15
Team metrics gotten for game ID = 0020200660, index = 16
Team metrics gotten for game ID = 0020200

In [73]:
# Define the columns to fill
trad_cols = [
    'TRAD_PTS', 'TRAD_FGM', 'TRAD_FGA', 'TRAD_FG_PCT', 'TRAD_3PM',
       'TRAD_3PA', 'TRAD_3P_PCT', 'TRAD_FTM', 'TRAD_FTA', 'TRAD_FT_PCT',
       'TRAD_OREB', 'TRAD_DREB', 'TRAD_REB', 'TRAD_AST', 'TRAD_TOV',
       'TRAD_STL', 'TRAD_BLK', 'TRAD_PF', 'TRAD_PLUS_MINUS', 'TRAD_MIN'
]

# Merge the dataframes
team_metrics5 = pd.merge(
    team_metrics_full,
    missing_trad_statsteam_metrics,
    on=['GAME_ID', 'TEAM_ID'],
    how='left',
    suffixes=('', '_from_metrics')
)

# Fill missing values from team_metrics
for col in trad_cols:
    team_metrics5[col] = team_metrics5[col].combine_first(team_metrics5[f"{col}_from_metrics"])
    team_metrics5.drop(columns=[f"{col}_from_metrics"], inplace=True)

# Merge the dataframes
team_metrics6 = pd.merge(
    team_metrics5,
    missing_trad_statsteam_metrics2,
    on=['GAME_ID', 'TEAM_ID'],
    how='left',
    suffixes=('', '_from_metrics')
)

# Fill missing values from team_metrics
for col in trad_cols:
    team_metrics6[col] = team_metrics6[col].combine_first(team_metrics6[f"{col}_from_metrics"])
    team_metrics6.drop(columns=[f"{col}_from_metrics"], inplace=True)

In [75]:
missing_values_count = team_metrics6.isnull().sum()
team_metrics6_columns = team_metrics6.columns
sum(team_metrics6_columns[missing_values_count > 0])

0

In [76]:
team_metrics6.to_csv('team_metrics_final.csv', index=False)